![NVIDIA Logo](images/nvidia.png)

# Control Message Tensors

In this notebook you will learn how to work with control message tensors. Control message tensors are efficient data structures for storing and managing large, multi-dimensional arrays within pipeline messages. They are particularly crucial for inference tasks, allowing seamless integration of input data for machine learning models and facilitating the storage of model outputs as messages flow through pipeline stages.

---

## Objectives

By the time you complete this notebook you will be able to:

- Understand the value of Morpheus providing support for tensor data.
- Create `TensorMemory` messages, and add them to `ControlMessage` messages.
- Have an intuition about sequence ID tensors, which are required in many inference tasks.

---

## Imports

In [1]:
import cupy as cp
from morpheus.messages import ControlMessage, TensorMemory

---

## Working With Tensors

Tensors, as the term suggests, are often large and multi-dimensional. In machine learning, tensor data is essential, and while we haven't explored it in this workshop yet, Morpheus pipelines are frequently designed for both machine learning training and inference using the data that passes through them.

If you're even slightly familiar with machine learning, you’ll know that high-dimensional floating-point data is not well supported by traditional dataframe structures. This is the primary reason why Morpheus includes optimized and highly performant support for tensors.

---

## Adding Tensors to Control Messages

Here we'll create some very simple tensor data, a 1-dimensional CuPy array.

In [2]:
tensor_data = cp.array([1, 2, 3, 4, 5])

As with metadata and task dictionaries, tensors are data attached to a control message. Thus we will also create here a control message.

In [3]:
# Create a ControlMessage
msg = ControlMessage()

To ensure Morpheus tensors are highly optimized, we need to allocate memory with a specific size to store them efficiently. This is achieved using the `morpheus.messages.TensorMemory` message type, which is always instantiated with a `count` parameter. The `count` represents the first dimension of the tensor's shape, defining how many elements it can hold along that axis.

In [4]:
count = tensor_data.shape[0]

tensor_memory = TensorMemory(count=count)

Having created a `TensorMemory` message of the appropriate size, we can now store a tensor within it with the `set_tensor` method which expects both a string label for the tensor data, along with the tensor data itself.

In [5]:
tensor_memory.set_tensor("my_tensor", tensor_data)

We can now use a control message's `tensors` method to set our tensor to the control message.

In [6]:
msg.tensors(tensor_memory)

If used without arguments, `tensors()` will act as a getter, and with it we can also get any tensors added to a control message.

In [7]:
msg.tensors().get_tensors()

{'my_tensor': array([1, 2, 3, 4, 5])}

In [8]:
msg.tensors().get_tensor("my_tensor")

array([1, 2, 3, 4, 5])

---

## Creating Sequence ID Tensors

In the next notebook we will be looking at using Morpheus stages that can perform inference. In doing so, we are going to need to provide the stage with **sequence ID tensors**.

It's beyond the scope of this workshop to take a very deep dive into inference topics like sequence IDs, however, since we will be providing them explicitly to Morpheus stages in later notebooks, we'd like you to have an intuition for what they are and how they are created.

To that end we will walk through a simple example of creating sequence IDs to accompany a collection of texts intended to be sent to a machine learning model for inference.

We'll begin by defining a list of texts. Let's assume that we have a machine learning model of some kind and we would like to send these texts in batch to the machine learning model to perform inference.

In [9]:
texts = [
    "Hello world",
    "Morpheus is great",
    "AI and ML are fascinating"
]

In a typical machine learning setting, text is tokenized before being sent to a model for inference. In our simplified example, we split each string into tokens based on spaces (`" "`).Once tokenized, our data forms a 2-dimensional tensor of tokens, where each row corresponds to a different input sequence.

In [10]:
# Tokenize (simplified for illustration)
tokenized = [text.split() for text in texts]

In [11]:
tokenized

[['Hello', 'world'],
 ['Morpheus', 'is', 'great'],
 ['AI', 'and', 'ML', 'are', 'fascinating']]

However, when working with machine learning models, particularly in inference pipelines, it’s common to create a sequence ID tensor to accompany the token tensor. This tensor provides structural information about the sequences being processed.

We need sequence IDs for the following reasons:
- Sequence IDs help models differentiate between input sequences in batched inference.
- They ensure proper attention mechanisms when using models like transformers.
- They allow padding so that all sequences have the same shape without losing information.

Sequence IDs are constructed in the following way:
1. Match the first dimension to the number of sequences (rows) in the tokenized input.
1. Match the second dimension to the longest sequence in the batch (padded if necessary).
1. Assign an ID to each token based on its sequence (same value per row, padding as `0`)


In the code below we:
- Find the longest sequence length (`max_length`).
- Generate a sequence ID tensor that assigns an ID per sequence, ensuring proper padding.

In [12]:
max_length = max(len(tokens) for tokens in tokenized)

In [13]:
max_length

5

In [14]:
seq_ids = cp.array([[i+1 if j < len(tokens) else 0 for j in range(max_length)] for i, tokens in enumerate(tokenized)])

In [15]:
print("Sequence IDs shape:", seq_ids.shape)
print("Sequence IDs:\n", seq_ids)

Sequence IDs shape: (3, 5)
Sequence IDs:
 [[1 1 0 0 0]
 [2 2 2 0 0]
 [3 3 3 3 3]]


Here, each row corresponds to a sequence, and each column represents a token slot:

- `1 1 0 0 0` → First sequence, padded after two tokens.
- `2 2 2 0 0` → Second sequence, padded after three tokens.
- `3 3 3 3 3` → Third sequence, fully occupying five slots.

Finally, we store the sequence ID tensor in Morpheus `TensorMemory`, making it available for further processing in the pipeline:

In [16]:
tensor_memory = TensorMemory(count=len(seq_ids))
tensor_memory.set_tensor("seq_ids", seq_ids)

In [ ]:
# (write)
# 1. cupy.array 생성
# 2. array.shape[0]을 count로 전달해서 TensorMemory 생성
# 3. TensorMemory의 set_tesnor로 "key"와 함께 할당
# 4. ControlMessage에 cm.tensors(TensorMemory)로 주입

# (read)
# TensorMemory = cm.tensors()
# TensorMemory.get_tensors()
# TensorMemory.get_tensors("key")